In [70]:
import glob
import numpy as np
import pandas as pd
from redteam.utils.data_utils import read_json

In [71]:
DATA_DIR = "/data/group_data/rl/datasets/redteaming/redteaming_evals/mistral_evals/2024.09.14/**/**.json"
fnames = glob.glob(DATA_DIR)
# fname = fnames[0]

In [72]:
# fnames = [fnames[0]]

In [73]:
fnames

['/data/group_data/rl/datasets/redteaming/redteaming_evals/mistral_evals/2024.09.14/23-00-1726369235/openai_sft_trained_attacker_untrained_defender_mistral_evals.json',
 '/data/group_data/rl/datasets/redteaming/redteaming_evals/mistral_evals/2024.09.14/23-37-1726371422/jailbreakbench_sft_trained_attacker_untrained_defender_mistral_evals.json']

In [74]:
# fname

In [75]:
fnames

['/data/group_data/rl/datasets/redteaming/redteaming_evals/mistral_evals/2024.09.14/23-00-1726369235/openai_sft_trained_attacker_untrained_defender_mistral_evals.json',
 '/data/group_data/rl/datasets/redteaming/redteaming_evals/mistral_evals/2024.09.14/23-37-1726371422/jailbreakbench_sft_trained_attacker_untrained_defender_mistral_evals.json']

In [76]:
def get_metrics(fname):
    data = read_json(fname)
    # get rewards
    rewards = []    
    for d in data:
        rewards.append(d["judge"]['rewards'])
    rewards = np.array(rewards)
    num_evals = len(rewards)
    num_jailbreaks = np.sum(np.any(rewards == 1., axis=1))
    return rewards,np.sum(rewards, axis=0), num_jailbreaks, num_evals 
    

In [77]:
agg_data = {}
for fname in fnames:
    _, rew_turns, n_jbs, num_evals=get_metrics(fname)
    agg_data[fname.split("/")[-1].split(".json")[0]] = rew_turns, n_jbs, num_evals

In [78]:
agg_data

{'openai_sft_trained_attacker_untrained_defender_mistral_evals': (array([2., 4., 8.]),
  8,
  15),
 'jailbreakbench_sft_trained_attacker_untrained_defender_mistral_evals': (array([ 5., 19., 43.]),
  49,
  100)}

In [79]:
# fname_key = "openai_rwr_trained_attacker_trained_defender_multilabel_value_function_evals"

data = []


for fname_key in sorted(agg_data.keys()):
    dataset = fname_key.split("_")[0]
    attacker_ft = fname_key.split("_trained_attacker")[0].split("_")[-1]
    defender_vf = fname_key.split("untrained_defender_")[-1].split("_value_function_evals")[0]
    rews = agg_data[fname_key][0]
    n_jbs = agg_data[fname_key][1]
    num_evals = agg_data[fname_key][2]
    data.append({
        'Dataset': dataset,
        'Attacker trained': attacker_ft,
        'Defender untrained': defender_vf,
        'Rewards': rews,
        'Num jailbreaks': n_jbs,
        'Num evals': num_evals
    })

df = pd.DataFrame(data)

In [80]:
df

,Dataset,Attacker trained,Defender untrained,Rewards,Num jailbreaks,Num evals
0,jailbreakbench,sft,mistral_evals,"[5.0, 19.0, 43.0]",49,100
1,openai,sft,mistral_evals,"[2.0, 4.0, 8.0]",8,15
